# 3_21 — ML Label Substrate Refresh (training_frame_v2_multi_date)

**ML FREEZE — owner directive 2026-11-27:** this task produces **labels only**. No training, no scoring,
no consumer in any decision path. ML-driven decisions stay frozen until 2026-12-01 (lift date set by owner, 2026-11-27 ruling).

Anti-leakage contract (structural, not policy): every feature is as-of `feature_date` (t-1); the label is
the band at `label_date` (t). Rebuilds `crm_ingestion.ml_prototype.training_frame_v2_multi_date` from
`gold.customer_health_history` — consecutive-date pairs per lead, NO DATA pairs excluded.


In [ ]:
from pyspark.sql import functions as F, Window
h = (spark.table('crm_ingestion.gold.customer_health_history')
     .withColumn('rn', F.row_number().over(Window.partitionBy('LEAD_ID','run_date')
              .orderBy(F.col('SOURCE_UPDATED_AT').desc()))).filter('rn = 1').drop('rn'))
w = Window.partitionBy('LEAD_ID').orderBy('run_date')
feats = ['DAYS_SINCE_LAST_EMAIL','DAYS_SINCE_LAST_LOGIN','DAYS_SINCE_LAST_MEETING',
         'DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT','DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER',
         'TOTAL_MISSED_CALLS','SENTIMENTS_LAST_30_DAYS','SLACK_ENGAGED_FLAG',
         'PLATFORM_ENGAGED_FLAG','MEETING_ENGAGED_FLAG','ENGAGEMENT_PATTERN',
         'BASE_ENGAGEMENT_SCORE','FINAL_SCORE','HEALTH_BAND','CONTRACT_VALUE']
p = h
for c in feats:
    p = p.withColumn(f'prev_{c}', F.lag(F.col(c)).over(w))
p = (p.withColumn('prev_run_date', F.lag('run_date').over(w))
      .withColumn('gap_days', F.datediff('run_date','prev_run_date')))
_ord = lambda col: (F.when(F.col(col)=='GOOD',3).when(F.col(col)=='AVERAGE',2)
                     .when(F.col(col)=='POOR',1))
p = (p.withColumn('band_ord', _ord('HEALTH_BAND'))
      .withColumn('prev_band_ord', _ord('prev_HEALTH_BAND')))
clean = (p.filter(F.col('band_ord').isNotNull() & F.col('prev_band_ord').isNotNull())
          .withColumn('label', F.when(F.col('band_ord')>F.col('prev_band_ord'),'UP')
              .when(F.col('band_ord')<F.col('prev_band_ord'),'DOWN').otherwise('STABLE')))
frame = (clean.select(F.monotonically_increasing_id().alias('pair_id'),
    'LEAD_ID','LEAD_EMAIL',
    F.col('prev_run_date').alias('feature_date'), F.col('run_date').alias('label_date'),
    'gap_days','label', *[F.col(f'prev_{c}') for c in feats])
    .withColumnRenamed('prev_HEALTH_BAND','prev_band'))
spark.sql('CREATE SCHEMA IF NOT EXISTS crm_ingestion.ml_prototype')
frame.write.mode('overwrite').saveAsTable('crm_ingestion.ml_prototype.training_frame_v2_multi_date')
chk = spark.table('crm_ingestion.ml_prototype.training_frame_v2_multi_date')
print('frame rows:', chk.count())
print('labels:', {r['label']: r['count'] for r in chk.groupBy('label').count().collect()})
print('ML freeze: labels only — no scoring, no decisions, no other consumers.')
